In [1]:
import sys
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [3]:
from src.etl.loader import load_all_datasets

In [4]:
datasets = load_all_datasets()

print(f"Loaded {len(datasets)} datasets")

for name, df in datasets.items():
    print(f"{name:<20} {df.shape}")

RAW PATH: C:\Users\panka\OneDrive\Desktop\Nifty100_Project\data\raw
SUPPORTING PATH: C:\Users\panka\OneDrive\Desktop\Nifty100_Project\data\raw\supporting datasets
Loaded 12 datasets
companies            (92, 12)
profitandloss        (1276, 15)
balancesheet         (1312, 13)
cashflow             (1187, 7)
analysis             (20, 6)
documents            (1585, 4)
prosandcons          (16, 4)
financial_ratios     (1184, 16)
market_cap           (552, 9)
peer_groups          (56, 4)
sectors              (92, 6)
stock_prices         (5520, 9)


In [5]:
validation_results = []

def log_result(rule, severity, table, issue, failed_rows):
    validation_results.append({
        "Rule": rule,
        "Severity": severity,
        "Table": table,
        "Issue": issue,
        "Failed Rows": int(failed_rows)
    })

In [6]:
def log_result(rule, severity, table, issue, failed_rows):
    validation_results.append({
        "Rule": rule,
        "Severity": severity,
        "Table": table,
        "Issue": issue,
        "Failed Rows": failed_rows
    })

In [7]:
datasets = load_all_datasets()

print(datasets["companies"].columns.tolist())

RAW PATH: C:\Users\panka\OneDrive\Desktop\Nifty100_Project\data\raw
SUPPORTING PATH: C:\Users\panka\OneDrive\Desktop\Nifty100_Project\data\raw\supporting datasets
['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']


In [8]:
datasets["companies"].head()

,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,46.0,34.90
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,9.0,8.59
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,11.6,13.64
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,96.5,14.70
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd\n,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,12.9,18.10


Cell 7 — DQ-01 (Companies PK)

In [9]:
for name, df in datasets.items():
    print("="*70)
    print(name)
    print(df.columns.tolist())

companies
['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']
profitandloss
['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit', 'opm_percentage', 'other_income', 'interest', 'depreciation', 'profit_before_tax', 'tax_percentage', 'net_profit', 'eps', 'dividend_payout']
balancesheet
['id', 'company_id', 'year', 'equity_capital', 'reserves', 'borrowings', 'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip', 'investments', 'other_asset', 'total_assets']
cashflow
['id', 'company_id', 'year', 'operating_activity', 'investing_activity', 'financing_activity', 'net_cash_flow']
analysis
['id', 'company_id', 'compounded_sales_growth', 'compounded_profit_growth', 'stock_price_cagr', 'roe']
documents
['id', 'company_id', 'Year', 'Annual_Report']
prosandcons
['id', 'company_id', 'pros', 'cons']
financial_ratios
['id', 'company_id', 'year', 'net_pr

In [10]:
companies = datasets["companies"]

duplicate_ids = companies["id"].duplicated().sum()

log_result(
    rule="DQ-01",
    severity="CRITICAL",
    table="companies",
    issue="Duplicate primary key (id)",
    failed_rows=duplicate_ids
)

print(f"Duplicate IDs found: {duplicate_ids}")

Duplicate IDs found: 0


In [11]:
pd.DataFrame(validation_results)

,Rule,Severity,Table,Issue,Failed Rows
0,DQ-01,CRITICAL,companies,Duplicate primary key (id),0


DQ-02 — Composite Primary Key (company_id + year)

In [12]:
tables_to_check = [
    "profitandloss",
    "balancesheet",
    "cashflow",
    "financial_ratios",
    "market_cap"
]

for table in tables_to_check:

    df = datasets[table]

    dup = df[df.duplicated(
        subset=["company_id","year"],
        keep=False
    )]

    duplicate_rows = len(dup)

    duplicate_companies = dup["company_id"].unique().tolist()

    log_result(
        rule="DQ-02",
        severity="CRITICAL",
        table=table,
        issue="Duplicate (company_id, year)",
        failed_rows=duplicate_rows
    )

    print(f"\n{table}")
    print("Duplicate rows :", duplicate_rows)
    print("Companies      :", duplicate_companies)


profitandloss
Duplicate rows : 26
Companies      : ['ADANIPORTS']

balancesheet
Duplicate rows : 138
Companies      : ['ASIANPAINT', 'PNB', 'POWERGRID', 'TECHM']

cashflow
Duplicate rows : 46
Companies      : ['ABB', 'BAJAJ-AUTO']

financial_ratios
Duplicate rows : 202
Companies      : ['ABB', 'ADANIPORTS', 'ASIANPAINT', 'BAJAJ-AUTO', 'PNB', 'POWERGRID', 'TECHM']

market_cap
Duplicate rows : 0
Companies      : []


DQ-03 Foreign Key Integrity

In [13]:
master_ids = set(datasets["companies"]["id"])

fk_tables = [
    "profitandloss",
    "balancesheet",
    "cashflow",
    "financial_ratios",
    "market_cap",
    "sectors"
]

for table in fk_tables:

    df = datasets[table]

    invalid = df.loc[
        ~df["company_id"].isin(master_ids),
        "company_id"
    ]

    failed_rows = len(invalid)

    missing_companies = sorted(invalid.unique())

    log_result(
        rule="DQ-03",
        severity="CRITICAL",
        table=table,
        issue="Foreign key company_id missing in companies table",
        failed_rows=failed_rows
    )

    print("\n" + "="*60)
    print(table)
    print(f"Missing FK Rows : {failed_rows}")
    print(f"Missing Companies : {missing_companies}")


profitandloss
Missing FK Rows : 99
Missing Companies : ['ULTRACEMCO', 'UNIONBANK', 'UNITDSPR', 'VBL', 'VEDL', 'WIPRO', 'ZOMATO', 'ZYDUSLIFE']

balancesheet
Missing FK Rows : 85
Missing Companies : ['ULTRACEMCO', 'UNIONBANK', 'UNITDSPR', 'VEDL', 'WIPRO', 'ZOMATO', 'ZYDUSLIFE']

cashflow
Missing FK Rows : 96
Missing Companies : ['AGTL', 'ULTRACEMCO', 'UNIONBANK', 'UNITDSPR', 'VBL', 'VEDL', 'WIPRO', 'ZOMATO', 'ZYDUSLIFE']

financial_ratios
Missing FK Rows : 24
Missing Companies : ['ULTRACEMCO', 'UNIONBANK']

market_cap
Missing FK Rows : 0
Missing Companies : []

sectors
Missing FK Rows : 0
Missing Companies : []


In [14]:
datasets["balancesheet"].columns.tolist()

['id',
 'company_id',
 'year',
 'equity_capital',
 'reserves',
 'borrowings',
 'other_liabilities',
 'total_liabilities',
 'fixed_assets',
 'cwip',
 'investments',
 'other_asset',
 'total_assets']

DQ-04 Balance Sheet Validation

In [15]:
bs = datasets["balancesheet"].copy()

bs["calculated_liabilities"] = (
    bs["equity_capital"] +
    bs["reserves"] +
    bs["borrowings"] +
    bs["other_liabilities"]
)

bs["difference_pct"] = (
    abs(bs["total_assets"] - bs["calculated_liabilities"])
    / bs["total_assets"]
) * 100

failed = (bs["difference_pct"] > 1).sum()

log_result(
    rule="DQ-04",
    severity="WARNING",
    table="balancesheet",
    issue="Balance Sheet mismatch >1%",
    failed_rows=failed
)

print("Rows failing balance check:", failed)

Rows failing balance check: 4


In [16]:
bs[bs["difference_pct"] > 1][[
    "company_id",
    "year",
    "total_assets",
    "calculated_liabilities",
    "difference_pct"
]].head(10)

,company_id,year,total_assets,calculated_liabilities,difference_pct
13,ADANIENSOL,Mar 2014,0,0.1,inf
211,BEL,Mar 2017,77,76.0,1.298701
590,INDIGO,Mar 2013,26,26.4,1.538462
591,INDIGO,Mar 2014,35,34.4,1.714286


In [17]:
pd.DataFrame(validation_results)

,Rule,Severity,Table,Issue,Failed Rows
0,DQ-01,CRITICAL,companies,Duplicate primary key (id),0
1,DQ-02,CRITICAL,profitandloss,"Duplicate (company_id, year)",26
2,DQ-02,CRITICAL,balancesheet,"Duplicate (company_id, year)",138
3,DQ-02,CRITICAL,cashflow,"Duplicate (company_id, year)",46
4,DQ-02,CRITICAL,financial_ratios,"Duplicate (company_id, year)",202
5,DQ-02,CRITICAL,market_cap,"Duplicate (company_id, year)",0
6,DQ-03,CRITICAL,profitandloss,Foreign key company_id missing in companies table,99
7,DQ-03,CRITICAL,balancesheet,Foreign key company_id missing in companies table,85
8,DQ-03,CRITICAL,cashflow,Foreign key company_id missing in companies table,96
9,DQ-03,CRITICAL,financial_ratios,Foreign key company_id missing in companies table,24


DQ-5 - OPM Cross Check

In [18]:
pl = datasets["profitandloss"].copy()

pl["calculated_opm"] = (
    pl["operating_profit"] / pl["sales"]
) * 100

pl["difference"] = (
    pl["calculated_opm"] - pl["opm_percentage"]
).abs()

failed = (pl["difference"] > 1).sum()

log_result(
    rule="DQ-05",
    severity="WARNING",
    table="profitandloss",
    issue="OPM percentage mismatch (>1%)",
    failed_rows=failed
)

print("Rows failing OPM check:", failed)

Rows failing OPM check: 234


In [19]:
pl[pl["difference"] > 1][[
    "company_id",
    "year",
    "sales",
    "operating_profit",
    "opm_percentage",
    "calculated_opm",
    "difference"
]].head(10)

,company_id,year,sales,operating_profit,opm_percentage,calculated_opm,difference
23,ADANIENSOL,Mar 2024,16607,5711.0,30.0,34.389113,4.389113
146,AXISBANK,Mar 2013,27183,8313.0,1353.0,30.581614,1322.418386
147,AXISBANK,Mar 2014,30641,9644.0,2307.0,31.474169,2275.525831
148,AXISBANK,Mar 2015,35479,11127.0,3097.0,31.362214,3065.637786
149,AXISBANK,Mar 2016,40988,13367.0,3466.0,32.611984,3433.388016
150,AXISBANK,Mar 2017,44542,23808.0,-5715.0,53.450676,5768.450676
151,AXISBANK,Mar 2018,45780,28895.0,-10277.0,63.117082,10340.117082
152,AXISBANK,Mar 2019,54986,27155.0,-5447.0,49.385298,5496.385298
153,AXISBANK,Mar 2020,62635,35066.0,-9859.0,55.984673,9914.984673
154,AXISBANK,Mar 2021,63346,31749.0,-2510.0,50.119976,2560.119976


DQ-06 — Positive Sales

In [20]:
negative_sales = (pl["sales"] <= 0).sum()

log_result(
    rule="DQ-06",
    severity="CRITICAL",
    table="profitandloss",
    issue="Sales must be greater than zero",
    failed_rows=negative_sales
)

print("Rows with non-positive sales:", negative_sales)

Rows with non-positive sales: 1


In [21]:
pd.DataFrame(validation_results)

,Rule,Severity,Table,Issue,Failed Rows
0,DQ-01,CRITICAL,companies,Duplicate primary key (id),0
1,DQ-02,CRITICAL,profitandloss,"Duplicate (company_id, year)",26
2,DQ-02,CRITICAL,balancesheet,"Duplicate (company_id, year)",138
3,DQ-02,CRITICAL,cashflow,"Duplicate (company_id, year)",46
4,DQ-02,CRITICAL,financial_ratios,"Duplicate (company_id, year)",202
5,DQ-02,CRITICAL,market_cap,"Duplicate (company_id, year)",0
6,DQ-03,CRITICAL,profitandloss,Foreign key company_id missing in companies table,99
7,DQ-03,CRITICAL,balancesheet,Foreign key company_id missing in companies table,85
8,DQ-03,CRITICAL,cashflow,Foreign key company_id missing in companies table,96
9,DQ-03,CRITICAL,financial_ratios,Foreign key company_id missing in companies table,24


In [23]:
for name, df in datasets.items():
    print("="*80)
    print(name.upper())
    print(df.columns.tolist())

COMPANIES
['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']
PROFITANDLOSS
['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit', 'opm_percentage', 'other_income', 'interest', 'depreciation', 'profit_before_tax', 'tax_percentage', 'net_profit', 'eps', 'dividend_payout']
BALANCESHEET
['id', 'company_id', 'year', 'equity_capital', 'reserves', 'borrowings', 'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip', 'investments', 'other_asset', 'total_assets']
CASHFLOW
['id', 'company_id', 'year', 'operating_activity', 'investing_activity', 'financing_activity', 'net_cash_flow']
ANALYSIS
['id', 'company_id', 'compounded_sales_growth', 'compounded_profit_growth', 'stock_price_cagr', 'roe']
DOCUMENTS
['id', 'company_id', 'Year', 'Annual_Report']
PROSANDCONS
['id', 'company_id', 'pros', 'cons']
FINANCIAL_RATIOS
['id', 'company_id', 'year', 'net_pr